# 05 — Generate Datapoints: the degradation story, as math

Companion to [Chapter 11](../11-datapoints.md). Cell order: auth -> compute the time
window -> generate values -> write -> inspect the last few points to confirm the
ramp is real, not imagined.

In [ ]:
YOURNAME = "YOURNAME"  # [CHANGE]

import math
import random
from datetime import datetime, timedelta, timezone

from cognite.client import CogniteClient
from cognite.client.data_classes.data_modeling import NodeId

client = CogniteClient()
space = f"isp_{YOURNAME}_TRN"
SERIES = ["21-PT-2001", "21-TT-2001", "21-LT-2001", "21-FT-2002", "21-VT-2002", "21-PT-2003"]


## Step 1 -- the 720-hour trailing window, relative to now

This is why a Function beats a static CSV: the window always ends at "now," whenever
you call it.

In [ ]:
end = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
start = end - timedelta(hours=719)
timestamps = [start + timedelta(hours=i) for i in range(720)]
print("window:", start.isoformat(), "->", end.isoformat())

## Step 2 -- the value function

Four series get gentle sinusoidal noise (normal operation). Two -- `21-FT-2002`
(flow, falling) and `21-VT-2002` (vibration, rising) -- carry a `degrade` ramp over
the last 120 of 720 hours. This contrast (four flat, two trending) is what makes the
trend visually obvious rather than lost in noise.

In [ ]:
random.seed(20260724)  # fixed seed -- reproducible shape across every run/participant

def value(tag: str, i: int, n: int) -> float:
    t = i / max(n - 1, 1)
    noise = random.uniform(-1, 1)
    degrade = max(0.0, (i - (n - 120)) / 119.0) if i >= n - 120 else 0.0
    if tag == "21-PT-2001":
        return 12.0 + 0.4 * math.sin(2 * math.pi * t * 4) + 0.1 * noise
    if tag == "21-TT-2001":
        return 68.0 + 2.0 * math.sin(2 * math.pi * t * 3) + 0.3 * noise
    if tag == "21-LT-2001":
        return 52.0 + 6.0 * math.sin(2 * math.pi * t * 2) + 0.8 * noise
    if tag == "21-FT-2002":
        return 320.0 + 8.0 * noise - degrade * (320.0 - 268.0)
    if tag == "21-VT-2002":
        return 2.1 + 0.2 * noise + degrade * (7.4 - 2.1)
    if tag == "21-PT-2003":
        return 38.0 + 0.6 * noise - degrade * (38.0 - 33.5)
    return 0.0

vibration = [value("21-VT-2002", i, 720) for i in range(720)]
print("vibration, first 3 hours:", [round(v, 2) for v in vibration[:3]])
print("vibration, last 3 hours: ", [round(v, 2) for v in vibration[-3:]])
print("-> should ramp from ~2.1 to ~7.4 mm/s")

## Step 3 -- write, per series

Idempotent: inserting at an existing timestamp overwrites rather than duplicating, so
rerunning this cell is safe.

In [ ]:
for tag in SERIES:
    values = [value(tag, i, 720) for i in range(720)]
    points = [{"timestamp": ts, "value": val} for ts, val in zip(timestamps, values)]
    client.time_series.data.insert(points, instance_id=NodeId(space, tag))
    print(f"wrote {len(points)} points to {tag}")

## Step 4 -- verify by reading back the last few points

In [ ]:
df = client.time_series.data.retrieve_dataframe(instance_id=NodeId(space, "21-VT-2002"), start="5d-ago", end="now")
print(df.tail(10))
print("\nNow open 21-VT-2002 in Fusion's chart view over the last 5 days -- the ramp should be visually obvious.")

## Bridge to the Function

Package this into `GenerateDatapoints`: `SERIES`/`_value` unchanged (this logic has
no reason to differ inside a Function), `space` from `envVars` instead of a notebook
variable, and a returned dict (`series`, `points_per_series`, `total`, `window`)
instead of printed output. See
[Chapter 11, section 11.4](../11-datapoints.md#114-write-the-function-generatedatapoints).